# Chapter 7: Files and I/O

Companion notebook. Read the written chapter at [chapter.html](https://natrask.github.io/engr1050-fall2026/NewMaterial/Textbook/Ch07_files_io/chapter.html). Solutions at the bottom.

These cells write files into the current working directory. If you run this on Colab they live in the session sandbox and disappear when the session ends.


## Example 1: write a file

In [ ]:
with open("hello.txt", "w") as f:
    f.write("Hello, ENGR 1050!\n")
    f.write("Welcome to file I/O.\n")


## Example 2: read the whole file

In [ ]:
with open("hello.txt") as f:
    text = f.read()
print(text)
print(len(text), "characters")


## Example 3: read line by line

In [ ]:
with open("hello.txt") as f:
    for line in f:
        print(line.rstrip())


## Example 4: append

In [ ]:
with open("hello.txt", "a") as f:
    f.write("A new line at the end.\n")
with open("hello.txt") as f:
    print(f.read())


## Example 5: create a CSV from scratch

In [ ]:
import csv
rows = [
    ["name", "position", "rbi"],
    ["Schwarber", "DH", 119],
    ["Turner", "SS", 66],
    ["Harper", "1B", 64],
]
with open("players.csv", "w", newline="") as f:
    csv.writer(f).writerows(rows)
print("wrote players.csv")


## Example 6: read CSV with csv.reader

In [ ]:
import csv
with open("players.csv") as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        print(row)


## Example 7: read CSV with DictReader

In [ ]:
import csv
with open("players.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row["name"]
        rbi = int(row["rbi"])
        print(f"{name:12s} {rbi}")


## Example 8: read CSV with pandas

In [ ]:
import pandas as pd
df = pd.read_csv("players.csv")
print(df)
print(df["rbi"].sum())
print(df[df["rbi"] > 100])


## Example 9: write CSV from list of dicts

In [ ]:
import csv
players = [
    {"name": "Schwarber", "rbi": 119},
    {"name": "Turner",    "rbi": 66},
    {"name": "Harper",    "rbi": 64},
]
with open("players_out.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "rbi"])
    writer.writeheader()
    writer.writerows(players)
print("wrote players_out.csv")


## Example 10: file-not-found

In [ ]:
try:
    with open("not_there.txt") as f:
        text = f.read()
except FileNotFoundError:
    print("file is missing, using a default")
    text = ""
print(repr(text))


## Example 11: save a matplotlib figure

In [ ]:
import matplotlib.pyplot as plt
xs = list(range(11))
ys = [x*x for x in xs]
plt.figure(figsize=(6, 4))
plt.plot(xs, ys, "o-")
plt.xlabel("x")
plt.ylabel("x squared")
plt.title("Quadratic")
plt.savefig("quadratic.png", dpi=150, bbox_inches="tight")
plt.close()
import os
print("file exists:", os.path.exists("quadratic.png"))


## Example 12: NIST CSV loader

In [ ]:
# The stress-strain data from Lecture 3. NIST no longer hosts this file,
# so the course repo keeps a copy.
import csv
import urllib.request
url = "https://raw.githubusercontent.com/natrask/engr1050-fall2026/main/NewMaterial/_shared/Data/U15Al6XXX-T81_BatchB13R01T2.6921W12.71.csv"
local = "U15Al6XXX-T81.csv"
urllib.request.urlretrieve(url, local)
with open(local) as f:
    reader = csv.DictReader(f)
    points = []
    for row in reader:
        d = float(row["Displacement_(mm)"])
        F = float(row["Force_(kN)"])
        points.append((d, F))
print(len(points), "points")
print(points[:5])


---
## Solutions

### Problem 1: write 1..20, then sum from file

In [ ]:
with open("numbers.txt", "w") as f:
    for n in range(1, 21):
        f.write(f"{n}\n")

def sum_file(path):
    total = 0
    with open(path) as f:
        for line in f:
            total += int(line.strip())
    return total

print(sum_file("numbers.txt"))


### Problem 2: top RBI from players.csv

In [ ]:
import csv

def top_rbi(path):
    best = None
    best_rbi = -1
    with open(path) as f:
        for row in csv.DictReader(f):
            r = int(row["rbi"])
            if r > best_rbi:
                best_rbi = r
                best = row["name"]
    return best

print(top_rbi("players.csv"))


### Problem 3: the read-then-loop bug

In [ ]:
# After f.read(), the file pointer is at end, so the for loop sees nothing.
# Use either f.read() OR a line loop, not both.  And use `with`.
with open("hello.txt") as f:
    for line in f:
        print(line, end="")


### Problem 4: safe_open

In [ ]:
def safe_open(path):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError:
        return ""

print(repr(safe_open("hello.txt"))[:60])
print(repr(safe_open("does_not_exist.txt")))


### Problem 5: save sin plot, verify file

In [ ]:
import math, os
import matplotlib.pyplot as plt
N = 100
xs = [i * 2 * math.pi / (N - 1) for i in range(N)]
ys = [math.sin(x) for x in xs]
plt.figure(figsize=(6, 4))
plt.plot(xs, ys)
plt.xlabel("x"); plt.ylabel("sin(x)"); plt.title("sin from 0 to 2pi")
plt.savefig("sine.png", dpi=150, bbox_inches="tight")
plt.close()
print("sine.png exists:", os.path.exists("sine.png"))
